# Binary Logistic Regression 

Its a classification algorithm that puts two uses probability to classify.
## How It Works
- **Linear Equation:** It starts with a standard linear combination of inputs: z = w ⋅ X + b (weights times features plus bias).

- **Sigmoid Function:** It applies the sigmoid function \(g(z) = \frac{1}{1 + e^{-z}}\) to map any real-valued number into a probability value between 0 and 1.

- **Decision Threshold:** A threshold—usually 0.5—turns the predicted probability into a clear class label (e.g., values ≥ 0.5 mean class 1, and < 0.5 mean class 0).


## Problem 
Our e-commerce company is launching a new premium subscription tier next month. We have a dump of historical user activity data from our platform, and we need to figure out which users are actually going to buy this premium upgrade so marketing can target them. We also need to know what factors make someone want to upgrade.

In [89]:
# generate some data 

import numpy as np
import pandas as pd

np.random.seed(42)
n_users=1000
time_spent = np.random.normal(loc=25, scale=10, size=n_users).round(1)
items_in_cart = np.random.poisson(lam=3, size=n_users)
age = np.random.choice([np.nan, 22, 28, 35, 45, 60], size=n_users, p=[0.05, 0.2, 0.3, 0.25, 0.15, 0.05])
device_type = np.random.choice(['Mobile', 'Desktop', 'Tablet'], size=n_users)

# 2. Inject a strong mathematical relationship into the target!
# Log-odds formula: we make higher time and items heavily increase the chance of upgrading
log_odds = 0.08 * time_spent + 0.4 * items_in_cart - 3.5
probabilities = 1 / (1 + np.exp(-log_odds)) # Sigmoid function
clicked = np.random.binomial(n=1, p=probabilities)


df_new = pd.DataFrame({
    'user_id': np.arange(10001, 10001 + n_users),
    'age': age,
    'device_type': device_type,
    'time_spent_mins': np.abs(time_spent), # remove negatives
    'items_in_cart': items_in_cart,
    'churn_risk_score': np.random.uniform(1, 5, size=n_users).round(2),
    'historical_purchases': np.random.negative_binomial(n=2, p=0.3, size=n_users),
    'clicked_upgrade_banner': np.where(clicked == 1, 'Yes', 'No')
})

df=pd.DataFrame(df_new)
df.describe()


,user_id,age,time_spent_mins,items_in_cart,churn_risk_score,historical_purchases
count,1000.000000,958.000000,1000.000000,1000.000000,1000.00000,1000.000000
mean,10500.500000,32.835073,25.217900,2.974000,2.99244,4.763000
std,288.819436,10.195064,9.729528,1.684685,1.18389,4.071782
min,10001.000000,22.000000,0.300000,0.000000,1.01000,0.000000
25%,10250.750000,28.000000,18.500000,2.000000,1.96750,2.000000
50%,10500.500000,28.000000,25.250000,3.000000,2.97500,4.000000
75%,10750.250000,35.000000,31.500000,4.000000,4.05000,6.250000
max,11000.000000,60.000000,63.500000,9.000000,5.00000,29.000000


In [90]:
# Make time positive
# df["time_spent_mins"]=df["time_spent_mins"].abs()
# or 
df.loc[df["time_spent_mins"]<0, "time_spent_mins"]= df['time_spent_mins'] * -1 
df.describe()


,user_id,age,time_spent_mins,items_in_cart,churn_risk_score,historical_purchases
count,1000.000000,958.000000,1000.000000,1000.000000,1000.00000,1000.000000
mean,10500.500000,32.835073,25.217900,2.974000,2.99244,4.763000
std,288.819436,10.195064,9.729528,1.684685,1.18389,4.071782
min,10001.000000,22.000000,0.300000,0.000000,1.01000,0.000000
25%,10250.750000,28.000000,18.500000,2.000000,1.96750,2.000000
50%,10500.500000,28.000000,25.250000,3.000000,2.97500,4.000000
75%,10750.250000,35.000000,31.500000,4.000000,4.05000,6.250000
max,11000.000000,60.000000,63.500000,9.000000,5.00000,29.000000


In [91]:
df.to_csv('ecommerce_data.csv')

In [92]:
df=pd.read_csv("ecommerce_data.csv")
print(df.head(50))


    Unnamed: 0  user_id   age device_type  time_spent_mins  items_in_cart  \
0            0    10001  22.0     Desktop             30.0              1   
1            1    10002  22.0     Desktop             23.6              2   
2            2    10003  28.0      Mobile             31.5              1   
3            3    10004  35.0      Tablet             40.2              7   
4            4    10005  28.0     Desktop             22.7              6   
5            5    10006  22.0      Mobile             22.7              1   
6            6    10007  22.0      Mobile             40.8              1   
7            7    10008  28.0     Desktop             32.7              4   
8            8    10009  45.0     Desktop             20.3              5   
9            9    10010  28.0      Tablet             30.4              5   
10          10    10011  60.0      Mobile             20.4              0   
11          11    10012  28.0     Desktop             20.3              5   

In [93]:
df.info()
print(df.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Unnamed: 0              1000 non-null   int64  
 1   user_id                 1000 non-null   int64  
 2   age                     958 non-null    float64
 3   device_type             1000 non-null   str    
 4   time_spent_mins         1000 non-null   float64
 5   items_in_cart           1000 non-null   int64  
 6   churn_risk_score        1000 non-null   float64
 7   historical_purchases    1000 non-null   int64  
 8   clicked_upgrade_banner  1000 non-null   str    
dtypes: float64(3), int64(4), str(2)
memory usage: 70.4 KB
Unnamed: 0                 0
user_id                    0
age                       42
device_type                0
time_spent_mins            0
items_in_cart              0
churn_risk_score           0
historical_purchases       0
clicked_upgrade_banner     0
dty

In [94]:
# fill age
median_age= df["age"].median()
df["age"]=df["age"].fillna(median_age)


In [95]:
# encoding banner yes no
df["target"]=df["clicked_upgrade_banner"].map({"Yes":1,'No':0})
df

,Unnamed: 0,user_id,age,device_type,time_spent_mins,items_in_cart,churn_risk_score,historical_purchases,clicked_upgrade_banner,target
0,0,10001,22.0,Desktop,30.0,1,2.78,6,No,0
1,1,10002,22.0,Desktop,23.6,2,1.45,4,No,0
2,2,10003,28.0,Mobile,31.5,1,2.41,2,No,0
3,3,10004,35.0,Tablet,40.2,7,4.99,3,Yes,1
4,4,10005,28.0,Desktop,22.7,6,4.56,12,No,0
...,...,...,...,...,...,...,...,...,...,...
995,995,10996,28.0,Mobile,22.2,3,1.43,7,No,0
996,996,10997,60.0,Tablet,43.0,1,2.68,3,No,0
997,997,10998,22.0,Desktop,31.4,3,3.91,19,Yes,1
998,998,10999,28.0,Tablet,19.3,2,1.43,0,Yes,1


In [96]:
#drop unnecessary columns
df_cleaned = df.drop(columns=["user_id","clicked_upgrade_banner"])

df_cleaned.isnull().sum()

Unnamed: 0              0
age                     0
device_type             0
time_spent_mins         0
items_in_cart           0
churn_risk_score        0
historical_purchases    0
target                  0
dtype: int64

In [97]:
df_encoded =pd.get_dummies(df_cleaned,columns=["device_type"],drop_first=True)

df_encoded

,Unnamed: 0,age,time_spent_mins,items_in_cart,churn_risk_score,historical_purchases,target,device_type_Mobile,device_type_Tablet
0,0,22.0,30.0,1,2.78,6,0,False,False
1,1,22.0,23.6,2,1.45,4,0,False,False
2,2,28.0,31.5,1,2.41,2,0,True,False
3,3,35.0,40.2,7,4.99,3,1,False,True
4,4,28.0,22.7,6,4.56,12,0,False,False
...,...,...,...,...,...,...,...,...,...
995,995,28.0,22.2,3,1.43,7,0,True,False
996,996,60.0,43.0,1,2.68,3,0,False,True
997,997,22.0,31.4,3,3.91,19,1,False,False
998,998,28.0,19.3,2,1.43,0,1,False,True


In [98]:
X=df_encoded.drop(columns=["target"])
y=df_encoded["target"]
print(X.head(2))

   Unnamed: 0   age  time_spent_mins  items_in_cart  churn_risk_score  \
0           0  22.0             30.0              1              2.78   
1           1  22.0             23.6              2              1.45   

   historical_purchases  device_type_Mobile  device_type_Tablet  
0                     6               False               False  
1                     4               False               False  


In [99]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42)
()

()

In [100]:
#standard scalling 

from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()

x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)
print(f"Training shapes: {x_train_scaled.shape}, {y_train.shape}")
print(f"Testing shapes: {x_test_scaled.shape}, {y_test.shape}")



# standard scalar vs min max scalar

Training shapes: (800, 8), (800,)
Testing shapes: (200, 8), (200,)


In [101]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

model=LogisticRegression(class_weight='balanced', random_state=42)
model.fit(x_train_scaled,y_train)
y_pred=model.predict(x_test_scaled)

accuracy =accuracy_score(y_test,y_pred)
print (f" Accuracy {accuracy*100:.3f}% \n")
print("--- Classification Report ---")
print(classification_report(y_test, y_pred))
print(f"Confusion Matrix : {confusion_matrix(y_test,y_pred)}")


 Accuracy 63.000% 

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.69      0.63      0.66       113
           1       0.57      0.63      0.60        87

    accuracy                           0.63       200
   macro avg       0.63      0.63      0.63       200
weighted avg       0.64      0.63      0.63       200

Confusion Matrix : [[71 42]
 [32 55]]


In [103]:
# Extract weights and pair them with feature names
# Note: Use your specific feature list variable from your pipeline step
feature_names = x_train.columns

coefficients = model.coef_[0]

# Create a clean DataFrame to view them
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Weight (Coefficient)': coefficients
}).sort_values(by='Weight (Coefficient)', ascending=False)

print("--- Model Coefficients ---")
print(importance_df.to_string(index=False))


--- Model Coefficients ---
             Feature  Weight (Coefficient)
       items_in_cart              0.737996
     time_spent_mins              0.696737
  device_type_Tablet              0.082463
  device_type_Mobile              0.064898
          Unnamed: 0              0.039542
historical_purchases              0.027469
                 age             -0.049961
    churn_risk_score             -0.163331


The weights for items_in_cart (0.738) and time_spent_mins (0.697). They have the highest positive values by far, meaning they are the strongest drivers for pushing a user to buy the premium subscription. The model also correctly figured out that high churn_risk_score (-0.163) pulls the probability down.